In [51]:
import numpy as np
from datetime import datetime
import pandas as pd

In [52]:
path = '../data/raw/spx_dividend.xlsx'
spx = pd.read_excel(path)
print(spx)

    Unnamed: 0.1 Unnamed: 0  IN069 / DV105    IN060
0              0 2026-01-01         1.1701  80.0964
1              1 2026-01-02         1.1678  80.0965
2              2 2026-01-05         1.1608  80.1168
3              3 2026-01-06         1.1535  80.1062
4              4 2026-01-07         1.1579  80.1390
..           ...        ...            ...      ...
65            65 2026-04-02         1.2867  84.6980
66            66 2026-04-03         1.2866  84.6952
67            67 2026-04-06         1.2801  84.6385
68            68 2026-04-07         1.2791  84.6377
69            69 2026-04-08         1.2791  84.6377

[70 rows x 4 columns]


In [53]:
rename_map = {
    'Unnamed: 0': 'Date',
    'IN069 / DV105': 'Est Dividends Yield',
    'IN060': 'Est Dividends'
}

spx = spx.rename(columns=rename_map)
# spx = spx.drop(columns='Unnamed: 0.1')
spx['Date'] = pd.to_datetime(spx['Date'])
spx = spx.set_index('Date')
spx

,Unnamed: 0.1,Est Dividends Yield,Est Dividends
Date,,,
2026-01-01,0,1.1701,80.0964
2026-01-02,1,1.1678,80.0965
2026-01-05,2,1.1608,80.1168
2026-01-06,3,1.1535,80.1062
2026-01-07,4,1.1579,80.1390
...,...,...,...
2026-04-02,65,1.2867,84.6980
2026-04-03,66,1.2866,84.6952
2026-04-06,67,1.2801,84.6385


In [54]:
spx_div = pd.read_excel(path, sheet_name='SPXDIVAN')
spx_div = spx_div.rename(columns={'Unnamed: 0': 'Date'})
spx_div['Date'] = pd.to_datetime(spx_div['Date'])
spx = spx.drop(columns='Unnamed: 0.1')
spx_div = spx_div.set_index('Date')
spx_div

,Unnamed: 0.1,SPXDIVAN
Date,,
2025-12-22,0,0.41
2025-12-23,1,0.43
2025-12-24,2,0.45
2025-12-26,3,1.12
2025-12-29,4,1.32
...,...,...
2026-03-31,67,23.02
2026-04-01,68,23.27
2026-04-02,69,23.75


In [55]:
spx_div = spx_div.drop(columns = ['Unnamed: 0.1'])

In [56]:
spx_div

,SPXDIVAN
Date,
2025-12-22,0.41
2025-12-23,0.43
2025-12-24,0.45
2025-12-26,1.12
2025-12-29,1.32
...,...
2026-03-31,23.02
2026-04-01,23.27
2026-04-02,23.75


In [57]:
dividend_df = pd.concat([spx, spx_div], axis=1)
dividend_df

/var/folders/9_/_hcc1_5d3mbf6gs_r1f4gbsm0000gn/T/ipykernel_46242/3577312893.py:1: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  dividend_df = pd.concat([spx, spx_div], axis=1)


,Est Dividends Yield,Est Dividends,SPXDIVAN
Date,,,
2025-12-22,NaN,NaN,0.41
2025-12-23,NaN,NaN,0.43
2025-12-24,NaN,NaN,0.45
2025-12-26,NaN,NaN,1.12
2025-12-29,NaN,NaN,1.32
...,...,...,...
2026-04-02,1.2867,84.6980,23.75
2026-04-03,1.2866,84.6952,NaN
2026-04-06,1.2801,84.6385,24.25


In [58]:
dividend_df.isna().sum()

Est Dividends Yield    7
Est Dividends          7
SPXDIVAN               5
dtype: int64

In [59]:
# any better way to handle N.A.?
dividend_df['SPXDIVAN'] = dividend_df['SPXDIVAN'].ffill()
dividend_df = dividend_df.dropna()
dividend_df

,Est Dividends Yield,Est Dividends,SPXDIVAN
Date,,,
2026-01-01,1.1701,80.0964,1.87
2026-01-02,1.1678,80.0965,3.46
2026-01-05,1.1608,80.1168,3.48
2026-01-06,1.1535,80.1062,3.99
2026-01-07,1.1579,80.1390,4.03
...,...,...,...
2026-04-02,1.2867,84.6980,23.75
2026-04-03,1.2866,84.6952,23.75
2026-04-06,1.2801,84.6385,24.25


In [60]:
# the most important part of FVA
dividend_df['Dividends Yield'] = np.log(1 + dividend_df['Est Dividends Yield'] / 100)
dividend_df = dividend_df['Dividends Yield']

In [61]:
dividend_df.to_excel('../data/processed/spx_dividend.xlsx')